In [1]:
# %pip install -r "../requirements.txt"

In [2]:
import pandas as pd
import requests
from time import sleep

API_KEY = {'x-api-key': '114514'}
BASE_URL = "http://localhost:9999/v1"

s = requests.Session()
s.headers.update(API_KEY)

C:\Users\EFuser\AppData\Local\Temp\ipykernel_16876\1089218512.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
from time import sleep


# Rest of the code...

def post_order(s, type, quantity, action, ticker):
    r = s.post("http://localhost:9999/v1/orders?type="
               + type + "&quantity=" + str(quantity) + "&action=" + action + "&ticker=" + ticker)

    if r.status_code == 429:
        wait = r.json()["wait"]
        sleep(wait)

        post_order(s, type, quantity, action, ticker)
    elif r.status_code != 200:
        print(r.json())


def post_limit_order(s, type, quantity, action, ticker, price):
    r = s.post("http://localhost:9999/v1/orders?type="
               + type + "&quantity=" + str(quantity) + "&action=" + action + "&ticker=" + ticker + "&price=" + str(
        price))

    if r.status_code == 429:
        wait = r.json()["wait"]
        sleep(wait)

        post_limit_order(s, type, quantity, action, ticker, price)
    elif r.status_code != 200:
        print(r.json())


def handle_tender_offer(s):
    tenders = s.get(BASE_URL + '/tenders').json()
    if not tenders:
        print(tenders)
        return

    tender = tenders[0]

    sec = s.get(BASE_URL + '/securities', params={"ticker": tender["ticker"]}).json()[0]
    macd = calculate_macd(s, tender["ticker"])
    rsi = calculate_rsi(s, tender["ticker"])

    if tender['action'] == 'BUY' and macd == "BUY" and rsi > 60 and tender["price"] < sec['bid'] - 0.1:
        s.post(BASE_URL + '/tenders/' + str(tender['tender_id']))
    elif tender['action'] == 'SELL' and macd == "SELL" and rsi < 40 and tender["price"] > sec['ask'] + 0.1:
        s.post(BASE_URL + '/tenders/' + str(tender['tender_id']))


def close_positions(s, ticker):
    sec = s.get(BASE_URL + '/securities', params={"ticker": ticker}).json()
    pos = sec[0]['position']
    action = "SELL" if pos > 0 else "BUY"
    quantity = 1000000 if ticker == "USD" else 10000
    pos = abs(pos)
    
    while pos > 0:
        s.post(BASE_URL + '/orders', params={'type': 'MARKET', 'quantity': min(quantity, pos), 'action': action, 'ticker': ticker})
        pos -= quantity


def compare_long_short_trend(s, ticker):
    ohlc = pd.DataFrame(s.get(BASE_URL + '/securities/history', params={"ticker": ticker, "limit": 30}).json())
    ohlc = ohlc.iloc[::-1]

    ema3 = ohlc['close'].ewm(span=3).mean()
    ema9 = ohlc['close'].ewm(span=9).mean()

    if ema3.iloc[-1] > ema9.iloc[-1]:
        return "BUY"
    elif ema3.iloc[-1] < ema9.iloc[-1]:
        return "SELL"
    else:
        return "HOLD"


def calculate_macd(s, ticker):
    ohlc = pd.DataFrame(s.get(BASE_URL + '/securities/history', params={"ticker": ticker, "limit": 20}).json())
    ohlc = ohlc.iloc[::-1]
    
    ema12 = ohlc['close'].ewm(span=12).mean()
    ema26 = ohlc['close'].ewm(span=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    
    if macd.iloc[-1] > signal.iloc[-1]:
        return "BUY"
    elif macd.iloc[-1] < signal.iloc[-1]:
        return "SELL"
    else:
        return "HOLD"
    
    
def calculate_rsi(s, ticker):
    ohlc = pd.DataFrame(s.get(BASE_URL + '/securities/history', params={"ticker": ticker, "limit": 30}).json())
    ohlc = ohlc.iloc[::-1]

    delta = ohlc['close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    # Calculate average gain and loss 
    average_gain = gain.rolling(window=14).mean()
    average_loss = loss.rolling(window=14).mean()

    # Calculate RSI 
    rs = average_gain / average_loss.abs()
    rsi = 100 - (100 / (1 + rs))
    return rsi.iloc[-1]


def final_action(s):
    securities = s.get(BASE_URL + '/securities').json()
    
    usd = securities[1]
    hawk = securities[2]
    dove = securities[3]
    ritc = securities[4]
    ritu = securities[5]
    

In [4]:
r = s.get(BASE_URL + '/case')
tick = r.json()['tick']
print(tick)

while True:
    sec = s.get(BASE_URL + '/securities').json()

    # FX trading logic
    # usd = s.get(BASE_URL + '/securities', params={"ticker": "USD"}).json()[0]
    # macd = calculate_macd(s, "USD")
    # rsi = calculate_rsi(s, "USD")

    # if usd["position"] == 0:
    #     if macd == "BUY" and rsi > 80:
    #         for _ in range(10):
    #             s.post(BASE_URL + '/orders',
    #                    params={'type': 'MARKET', 'quantity': 1000000, 'action': 'BUY', 'ticker': 'USD'})
    #     elif macd == "SELL" and rsi < 20:
    #         for _ in range(10):
    #             s.post(BASE_URL + '/orders',
    #                    params={'type': 'MARKET', 'quantity': 1000000, 'action': 'SELL', 'ticker': 'USD'})
    # elif usd["position"] > 0:
    #     if macd == "SELL" and rsi < 35:
    #         close_positions(s, "USD")
    # elif usd["position"] < 0:
    #     if macd == "BUY" and rsi > 65:
    #         close_positions(s, "USD")

    # for security in sec:
    #     if security['position'] == 0:
    #         continue
    #         
    #     if security['ticker'] == "USD":
    #         close_positions(s, "USD")
    #         continue
    #         
    #     macd = calculate_macd(s, security['ticker'])
    #     rsi = calculate_rsi(s, security['ticker'])

        # if security['position'] > 0:
        #     if macd == "SELL" and rsi < 30:
        #         close_positions(s, security['ticker'])
        # elif security['position'] < 0:
        #     if macd == "BUY" and rsi > 70:
        #         close_positions(s, security['ticker'])

    # handle tender offer
    handle_tender_offer(s)

    tick = s.get(BASE_URL + '/case').json()['tick']
    # if tick > 295:
    # final_action(s)

    sleep(1)

20
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[

KeyError: 0